In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StructType, StringType
from delta.tables import DeltaTable
import requests
import json
import pandas as pd


In [0]:

dbutils.widgets.text("Catalog", "portfolio-activities")
dbutils.widgets.text("Schema", "github-api")
catalog = dbutils.widgets.get("Catalog")
schema = dbutils.widgets.get("Schema")

catalog = f"`{catalog}`"
schema = f"`{schema}`"

In [0]:
spark_issues_url = "https://api.github.com/repos/apache/spark/issues"
spark_pulls_url = "https://api.github.com/repos/apache/spark/pulls"

issues_payload = requests.get(spark_issues_url).json()
pulls_payload = requests.get(spark_pulls_url).json()

def get_df(url):
    payload = requests.get(url).json()
    pdf = pd.json_normalize(payload)

    spark_df = spark.createDataFrame(pdf)
    return spark_df

git_issues = get_df(spark_issues_url)
git_pulls = get_df(spark_pulls_url)


# Cannot us this because on Free Edition & function SparkContext isnt suported on serverless compute
# issues_rdd = spark.sparkContext.parallelize(
# [json.dumps(record) for record in response])

# issues_df = spark.read.json(issues_rdd)

In [0]:
def enrich_df(df):
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())
    return df


enriched_issues = enrich_df(git_issues)
enriched_pulls = enrich_df(git_pulls)

def array_to_string(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType): 
            print(field.name)
            df = df.withColumn(field.name,F.to_json(F.col(f"`{field.name}`").cast(ArrayType(StringType()))))
    df.printSchema()  
    return df

#Need to convert array to string as the nulls are causing issues with the table write
enriched_issues = array_to_string(enriched_issues)
enriched_pulls = array_to_string(enriched_pulls)

display(enriched_issues.limit(5))
display(enriched_pulls.limit(5))

In [0]:
#SCHEMA CHECK
required_columns = {"id", "title", "state", "created_at", "updated_at","number"}

def schema_check(df):
    columns_present = required_columns.issubset(df.columns)
    if not columns_present:
        raise ValueError('Schema Check Failed')
    else:
        print('Schema Check Passed')
    return

print('Schema Check Issues:')
schema_check(enriched_issues)
print('Schema Check Pulls:')
schema_check(enriched_pulls)

def data_quality_check(df):
    null_id = df.filter(df.id.isNull()).count()
    duplicate_id = df.groupBy("id").count().filter("count > 1").select("id")
    duplicate_id_count = duplicate_id.count()
    null_titles = df.filter(df.title.isNull()).count()

    #quality df output
    df_data = [("null_ids", null_id), ("duplicate_ids", duplicate_id_count), ("null_titles", null_titles)]
    df_columns = ["check", "result"]
    data_quality = spark.createDataFrame(df_data, df_columns)
    display(data_quality)

    #bad rows
    if null_id > 0 or duplicate_id_count > 0 or null_titles > 0:
        bad_df = df.filter(
            df.id.isNull() | 
            df.title.isNull() |
            df.id.isin([r.id for r in duplicate_id.collect()])
        )
        print('Bad rows:')
        display(bad_df) 
    return

print('Data Quality results issues:')
data_quality_check(enriched_issues)

print('Data Quality results pulls:')
data_quality_check(enriched_pulls)

In [0]:
def idempotent_write(df, table_name):
    if spark.catalog.tableExists(table_name):
        print('Appending Data .... performing indempotency logic....')
        existing_table = spark.table(table_name)
        latest_updated_at = existing_table.agg(F.max("updated_at")).collect()[0][0]
        existing_table = DeltaTable.forName(spark,table_name)
        print('Filtering new data based on latest updated_at timestamp......')
        new_issues = df.filter(F.col("updated_at") > latest_updated_at)
        new_issues = new_issues.withColumn("modified_timestamp",F.current_timestamp())
        #declaring columns to be used in set to preserve original ingestion_timestamp & created_at
        set_columns = {
            f"`{c}`": f"source.`{c}`"
            for c in new_issues.columns
            if c not in ["created_at", "ingestion_timestamp"]
        }
        print('Upserting Data.........')
        ( #adding upsert logic for any potential matched ids
            existing_table.alias("target").merge(
                source = new_issues.alias("source"),
                condition = "target.id = source.id"
            )
            .whenMatchedUpdate(set = set_columns)
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f'Data Appended: {new_issues.count()} rows to {table_name}')

    else:
        print(f"Table {table_name} does not exist, writing new table...")
        df = df.withColumn("modified_timestamp", F.current_timestamp())
        df.write.mode("overwrite").saveAsTable(table_name)
        print(f"Table {table_name} created")

In [0]:
idempotent_write(enriched_issues, f"{catalog}.{schema}.`tickets-spark_issues_bronze`")
idempotent_write(enriched_pulls, f"{catalog}.{schema}.`tickets-spark_pulls_bronze`")